# Train a Text-to-Image Model

This notebook demonstrates how to train a text-to-image model using the Hugging Face `diffusers` library.
We'll fine-tune a small Stable Diffusion model on a custom dataset.

## Overview
- Setup environment and dependencies
- Prepare a small custom dataset
- Load and configure the model
- Train the model
- Generate images from text prompts

## 1. Install Dependencies

In [ ]:
!pip install -q diffusers[torch] transformers accelerate datasets pillow torch torchvision
!pip install -q bitsandbytes  # For memory optimization
!pip install -q xformers  # For faster training (optional)

## 2. Import Libraries

In [ ]:
import os
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

from diffusers import StableDiffusionPipeline, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer
from datasets import load_dataset, Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from accelerate import Accelerator
from tqdm.auto import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 3. Configuration

In [ ]:
# Training configuration
config = {
    "model_id": "runwayml/stable-diffusion-v1-5",  # Base model
    "resolution": 512,  # Image resolution
    "train_batch_size": 1,  # Small batch size for limited GPU memory
    "num_epochs": 5,  # Number of training epochs
    "learning_rate": 1e-5,
    "gradient_accumulation_steps": 4,  # Simulate larger batch size
    "mixed_precision": "fp16",  # Use mixed precision for efficiency
    "output_dir": "./text-to-image-model",
    "seed": 42,
}

# Create output directory
os.makedirs(config["output_dir"], exist_ok=True)

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 4. Create a Small Custom Dataset

For demonstration, we'll create a synthetic dataset. In practice, you would use your own images and captions.

In [ ]:
def create_sample_dataset(num_samples=20):
    """
    Create a small synthetic dataset for demonstration.
    In practice, replace this with your actual dataset.
    """
    data = []
    
    # Sample text prompts
    prompts = [
        "a red circle on white background",
        "a blue square on white background",
        "a green triangle on white background",
        "a yellow star on white background",
        "a purple heart on white background",
    ]
    
    dataset_dir = Path("./sample_dataset")
    dataset_dir.mkdir(exist_ok=True)
    
    for i in range(num_samples):
        # Create simple geometric images
        img = Image.new('RGB', (512, 512), color='white')
        
        # Save image
        img_path = dataset_dir / f"image_{i}.png"
        img.save(img_path)
        
        # Get corresponding prompt
        prompt = prompts[i % len(prompts)]
        
        data.append({
            "image": str(img_path),
            "text": prompt
        })
    
    return data

# Alternative: Load from Hugging Face dataset
def load_hf_dataset():
    """
    Load a small public dataset from Hugging Face.
    Example: pokemon dataset
    """
    dataset = load_dataset("lambdalabs/pokemon-blip-captions", split="train[:50]")  # Just 50 samples
    return dataset

# Choose dataset source
USE_HF_DATASET = True  # Set to False to use synthetic dataset

if USE_HF_DATASET:
    print("Loading dataset from Hugging Face...")
    dataset = load_hf_dataset()
else:
    print("Creating synthetic dataset...")
    sample_data = create_sample_dataset()
    dataset = Dataset.from_list(sample_data)

print(f"Dataset size: {len(dataset)} samples")
print("\nSample entry:")
print(dataset[0])

## 5. Visualize Sample Data

In [ ]:
def show_samples(dataset, num_samples=4):
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 4))
    
    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        
        # Handle different dataset formats
        if isinstance(sample['image'], str):
            img = Image.open(sample['image'])
        else:
            img = sample['image']
        
        text = sample['text']
        
        axes[i].imshow(img)
        axes[i].set_title(text[:30] + "..." if len(text) > 30 else text, fontsize=8)
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(dataset)

## 6. Prepare Data Loader

In [ ]:
class TextImageDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, tokenizer, resolution=512):
        self.dataset = dataset
        self.tokenizer = tokenizer
        
        self.transforms = transforms.Compose([
            transforms.Resize(resolution, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(resolution),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),  # Normalize to [-1, 1]
        ])
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Load and process image
        if isinstance(item['image'], str):
            image = Image.open(item['image']).convert('RGB')
        else:
            image = item['image'].convert('RGB')
        
        image = self.transforms(image)
        
        # Tokenize text
        text = item['text']
        text_inputs = self.tokenizer(
            text,
            padding="max_length",
            max_length=self.tokenizer.model_max_length,
            truncation=True,
            return_tensors="pt",
        )
        
        return {
            "pixel_values": image,
            "input_ids": text_inputs.input_ids[0],
            "text": text,
        }

print("Dataset preparation complete.")

## 7. Load Pre-trained Model Components

In [ ]:
print("Loading model components...")

# Load tokenizer and text encoder
tokenizer = CLIPTokenizer.from_pretrained(
    config["model_id"], 
    subfolder="tokenizer"
)

text_encoder = CLIPTextModel.from_pretrained(
    config["model_id"], 
    subfolder="text_encoder"
)

# Load UNet (the main model we'll train)
unet = UNet2DConditionModel.from_pretrained(
    config["model_id"], 
    subfolder="unet"
)

# Load noise scheduler
noise_scheduler = DDPMScheduler.from_pretrained(
    config["model_id"], 
    subfolder="scheduler"
)

# Freeze text encoder (we only train the UNet)
text_encoder.requires_grad_(False)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

text_encoder.to(device)
unet.to(device)

print("Model loaded successfully!")

## 8. Setup Training

In [ ]:
# Create dataset and dataloader
train_dataset = TextImageDataset(dataset, tokenizer, config["resolution"])
train_dataloader = DataLoader(
    train_dataset, 
    batch_size=config["train_batch_size"], 
    shuffle=True
)

# Optimizer
optimizer = torch.optim.AdamW(
    unet.parameters(),
    lr=config["learning_rate"],
)

# Learning rate scheduler
from torch.optim.lr_scheduler import CosineAnnealingLR
lr_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=config["num_epochs"] * len(train_dataloader),
)

print(f"Training setup complete.")
print(f"Number of batches per epoch: {len(train_dataloader)}")
print(f"Total training steps: {config['num_epochs'] * len(train_dataloader)}")

## 9. Training Loop

In [ ]:
def train_model():
    unet.train()
    
    global_step = 0
    losses = []
    
    print("Starting training...\n")
    
    for epoch in range(config["num_epochs"]):
        epoch_loss = 0.0
        progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{config['num_epochs']}")
        
        for step, batch in enumerate(progress_bar):
            # Move batch to device
            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            
            # Get text embeddings
            with torch.no_grad():
                encoder_hidden_states = text_encoder(input_ids)[0]
            
            # Sample noise
            noise = torch.randn_like(pixel_values)
            bsz = pixel_values.shape[0]
            
            # Sample random timesteps
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps, (bsz,), 
                device=pixel_values.device
            ).long()
            
            # Add noise to images
            noisy_latents = noise_scheduler.add_noise(pixel_values, noise, timesteps)
            
            # Predict noise
            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            
            # Calculate loss
            loss = torch.nn.functional.mse_loss(model_pred, noise, reduction="mean")
            
            # Backpropagation
            loss.backward()
            
            # Update weights every gradient_accumulation_steps
            if (step + 1) % config["gradient_accumulation_steps"] == 0:
                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad()
            
            # Track metrics
            epoch_loss += loss.detach().item()
            losses.append(loss.detach().item())
            global_step += 1
            
            # Update progress bar
            progress_bar.set_postfix({"loss": loss.detach().item()})
        
        avg_epoch_loss = epoch_loss / len(train_dataloader)
        print(f"Epoch {epoch + 1} - Average Loss: {avg_epoch_loss:.4f}")
    
    return losses

# Train the model
training_losses = train_model()

print("\nTraining complete!")

## 10. Plot Training Loss

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(training_losses)
plt.xlabel('Training Step')
plt.ylabel('Loss')
plt.title('Training Loss Over Time')
plt.grid(True)
plt.show()

print(f"Final loss: {training_losses[-1]:.4f}")

## 11. Save the Trained Model

In [ ]:
print("Saving model...")

# Save UNet
unet.save_pretrained(os.path.join(config["output_dir"], "unet"))

print(f"Model saved to {config['output_dir']}")

## 12. Load Trained Model for Inference

In [ ]:
from diffusers import StableDiffusionPipeline

print("Loading trained model for inference...")

# Create pipeline with trained UNet
pipeline = StableDiffusionPipeline.from_pretrained(
    config["model_id"],
    unet=unet,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

pipeline = pipeline.to(device)

# Enable memory optimizations
if torch.cuda.is_available():
    pipeline.enable_attention_slicing()

print("Pipeline ready for inference!")

## 13. Generate Images from Text Prompts

In [ ]:
def generate_images(prompts, num_inference_steps=50, guidance_scale=7.5):
    """
    Generate images from text prompts using the trained model.
    
    Args:
        prompts: List of text prompts
        num_inference_steps: Number of denoising steps
        guidance_scale: Guidance scale for classifier-free guidance
    """
    images = []
    
    for prompt in prompts:
        print(f"Generating: {prompt}")
        
        with torch.no_grad():
            image = pipeline(
                prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=guidance_scale,
            ).images[0]
        
        images.append(image)
    
    return images

# Test prompts
test_prompts = [
    "a beautiful sunset over mountains",
    "a cute cat playing with a ball",
    "a futuristic city at night",
    "a fantasy castle in the clouds",
]

# Generate images
generated_images = generate_images(test_prompts)

print("\nGeneration complete!")

## 14. Visualize Generated Images

In [ ]:
def display_generated_images(prompts, images):
    num_images = len(images)
    cols = 2
    rows = (num_images + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(12, 6 * rows))
    axes = axes.flatten() if num_images > 1 else [axes]
    
    for idx, (prompt, image) in enumerate(zip(prompts, images)):
        axes[idx].imshow(image)
        axes[idx].set_title(prompt, fontsize=10, wrap=True)
        axes[idx].axis('off')
    
    # Hide empty subplots
    for idx in range(num_images, len(axes)):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

display_generated_images(test_prompts, generated_images)

## 15. Interactive Generation

In [ ]:
# Generate images from custom prompts
custom_prompt = "a red sports car on a beach"  # Change this to your desired prompt

print(f"Generating image for: '{custom_prompt}'")
custom_image = generate_images([custom_prompt])[0]

plt.figure(figsize=(8, 8))
plt.imshow(custom_image)
plt.title(custom_prompt)
plt.axis('off')
plt.show()

# Save generated image
output_path = os.path.join(config["output_dir"], "generated_image.png")
custom_image.save(output_path)
print(f"Image saved to: {output_path}")

## 16. Model Evaluation and Comparison

In [ ]:
# Compare with base model (before fine-tuning)
print("Loading base model for comparison...")

base_pipeline = StableDiffusionPipeline.from_pretrained(
    config["model_id"],
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)
base_pipeline = base_pipeline.to(device)

if torch.cuda.is_available():
    base_pipeline.enable_attention_slicing()

# Generate same prompt with both models
comparison_prompt = "a magical forest with glowing mushrooms"

print(f"\nGenerating with base model: '{comparison_prompt}'")
with torch.no_grad():
    base_image = base_pipeline(comparison_prompt, num_inference_steps=50).images[0]

print(f"Generating with fine-tuned model: '{comparison_prompt}'")
with torch.no_grad():
    finetuned_image = pipeline(comparison_prompt, num_inference_steps=50).images[0]

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(base_image)
axes[0].set_title(f"Base Model\n{comparison_prompt}", fontsize=10)
axes[0].axis('off')

axes[1].imshow(finetuned_image)
axes[1].set_title(f"Fine-tuned Model\n{comparison_prompt}", fontsize=10)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 17. Summary and Next Steps

### What We Accomplished:
1. ✅ Loaded and configured a pre-trained Stable Diffusion model
2. ✅ Prepared a custom dataset for training
3. ✅ Fine-tuned the model on our dataset
4. ✅ Generated images from text prompts
5. ✅ Compared base model vs fine-tuned model

### Next Steps:
- **Expand Dataset**: Use more images and diverse captions for better results
- **Longer Training**: Train for more epochs with a larger dataset
- **Hyperparameter Tuning**: Experiment with learning rates, batch sizes, etc.
- **Advanced Techniques**: Try LoRA, DreamBooth, or Textual Inversion
- **Evaluation Metrics**: Implement FID, CLIP score for quantitative evaluation

### Tips:
- Training text-to-image models requires significant GPU memory (8GB+ recommended)
- Use smaller batch sizes and gradient accumulation for limited GPU memory
- Consider using cloud platforms (Colab, Kaggle) for better GPU access
- For production use, train on thousands of high-quality image-text pairs

In [ ]:
print("="*60)
print("Text-to-Image Model Training Complete!")
print("="*60)
print(f"\nModel saved at: {config['output_dir']}")
print(f"Training epochs: {config['num_epochs']}")
print(f"Final training loss: {training_losses[-1]:.4f}")
print(f"\nYou can now use the trained model to generate images from text!")